# Exploring `agent.py` — The Stateful Wrapper

This notebook is a hands-on walkthrough of `liteagent.Agent` — the stateful class that wraps
the raw loop. If the loop notebook explored the **engine**, this one explores the **car**.

**Why two layers?**

The raw loop (`agent_loop`, `agent_loop_continue`) is stateless — you pass in context, it returns
an EventStream, you iterate events yourself. It's the engine.

The `Agent` class wraps the loop and manages:
- **Message history** — accumulates across turns, no manual context threading
- **Event subscription** — `subscribe(callback)` instead of manual `async for`
- **Steering + follow-up queues** — `steer()` and `follow_up()` with dequeue modes
- **Cancellation** — `abort()` with partial message preservation
- **State tracking** — `is_streaming`, `stream_message`, `pending_tool_calls`, `error`

This is the same two-layer design as pi-mono (`agent-loop.ts` + `agent.ts`).
Most consumers will use `Agent`. Advanced consumers may use the raw loop directly.

**What we'll cover:**
1. Setup + imports
2. Simplest prompt — string in, messages out
3. `subscribe()` — the primary consumer API
4. `prompt()` overloads — string, dict, list, images
5. State access — what you can inspect during and after a run
6. Multi-turn — why Agent is stateful
7. Steering — `steer()` mid-run
8. Follow-up — `follow_up()` after idle
9. Queue modes — one-at-a-time vs all
10. `continue_run()` — resume from context
11. `abort()` and partial preservation
12. `wait_for_idle()`
13. `reset()` vs `clear_messages()`
14. Configuration setters — mid-run changes
15. Error handling
16. `_default_convert_to_llm` — what it does
17. Testing across models
18. Real-world patterns
19. Summary

---

## 1. Setup

The Agent needs a model string (litellm format) and optionally tools, system prompt,
and a `convert_to_llm` function. Let's import everything and define our helpers.

In [1]:
from liteagent import Agent, Tool, ToolResult

# Default model for all examples
MODEL = "anthropic/claude-sonnet-4-6"


# convert_to_llm: strips our extras, keeps LLM-compatible fields.
# The Agent provides a default, but we'll define one explicitly so
# we can see exactly what it does (and override for provider quirks later).
def simple_convert(messages):
    result = []
    for m in messages:
        role = m.get("role")
        if role == "assistant":
            msg = {"role": "assistant"}
            if m.get("content"):
                msg["content"] = m["content"]
            if m.get("tool_calls"):
                msg["tool_calls"] = m["tool_calls"]
            if m.get("thinking_blocks"):
                msg["thinking_blocks"] = m["thinking_blocks"]
            if m.get("reasoning_content"):
                msg["reasoning_content"] = m["reasoning_content"]
            result.append(msg)
        elif role == "user":
            result.append({"role": "user", "content": m["content"]})
        elif role == "tool":
            content = m.get("content")
            if isinstance(content, list):
                text_parts = [b["text"] for b in content if b.get("type") == "text"]
                content = "\n".join(text_parts)
            result.append(
                {"role": "tool", "tool_call_id": m["tool_call_id"], "content": content}
            )
    return result


# Simple echo tool — reused across examples
async def echo_execute(tool_call_id, params, signal=None, on_update=None):
    return ToolResult(content=[{"type": "text", "text": params["message"]}])


echo_tool = Tool(
    name="echo",
    description="Echo back a message exactly",
    parameters={
        "type": "object",
        "properties": {
            "message": {"type": "string", "description": "The message to echo"}
        },
        "required": ["message"],
    },
    execute=echo_execute,
)

print("Setup complete.")

Setup complete.


## 2. Simplest prompt — string in, messages out

The absolute minimum: create an Agent, call `prompt("...")`, check `agent.messages`.

Unlike the raw loop (where you build `AgentContext` + `AgentConfig`, call `agent_loop`,
and iterate the EventStream yourself), the Agent does all of that internally.
`prompt()` blocks until the loop completes.

In [2]:
agent = Agent(
    model=MODEL,
    system_prompt="Be concise. One sentence max.",
    convert_to_llm=simple_convert,
)

await agent.prompt("What is 2 + 2?")



In [5]:
agent.messages

[{'role': 'user', 'content': 'What is 2 + 2?', 'timestamp': 1772757566207},
 {'role': 'assistant',
  'content': '4',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 25,
   'completion_tokens': 5,
   'total_tokens': 30,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772757567133}]

In [9]:
agent.state.messages

[{'role': 'user', 'content': 'What is 2 + 2?', 'timestamp': 1772757566207},
 {'role': 'assistant',
  'content': '4',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 25,
   'completion_tokens': 5,
   'total_tokens': 30,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772757567133}]

In [19]:
agent.state

AgentState(system_prompt='Be concise. One sentence max.', model='anthropic/claude-sonnet-4-6', thinking_level='off', tools=[], messages=[{'role': 'user', 'content': 'What is 2 + 2?', 'timestamp': 1772757566207}, {'role': 'assistant', 'content': '4', 'tool_calls': None, 'thinking_blocks': None, 'reasoning_content': None, 'provider_specific_fields': None, 'usage': {'prompt_tokens': 25, 'completion_tokens': 5, 'total_tokens': 30, 'cache_read_tokens': 0, 'cache_creation_tokens': 0}, 'stop_reason': 'stop', 'timestamp': 1772757567133}], is_streaming=False, stream_message=None, pending_tool_calls=set(), error=None)

In [22]:
agent.state.messages

[{'role': 'user', 'content': 'What is 2 + 2?', 'timestamp': 1772757566207},
 {'role': 'assistant',
  'content': '4',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 25,
   'completion_tokens': 5,
   'total_tokens': 30,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772757567133}]

In [23]:
simple_convert(agent.state.messages)

[{'role': 'user', 'content': 'What is 2 + 2?'},
 {'role': 'assistant', 'content': '4'}]

Two messages: the user prompt we sent, and the assistant's reply.
The Agent appended both to `agent.messages` automatically —
this is the key difference from the raw loop where you manage context yourself.

Let's look at the raw message objects:

In [24]:
# User message — our input, wrapped in a dict by prompt()
agent.messages[0]

{'role': 'user', 'content': 'What is 2 + 2?', 'timestamp': 1772757566207}

In [25]:
# Assistant message — enriched with usage, stop_reason, timestamp
agent.messages[1]

{'role': 'assistant',
 'content': '4',
 'tool_calls': None,
 'thinking_blocks': None,
 'reasoning_content': None,
 'provider_specific_fields': None,
 'usage': {'prompt_tokens': 25,
  'completion_tokens': 5,
  'total_tokens': 30,
  'cache_read_tokens': 0,
  'cache_creation_tokens': 0},
 'stop_reason': 'stop',
 'timestamp': 1772757567133}

In [27]:
# This is why we usd @property --> Read-only access. With @property, ust prevents replacing the state object itself.
agent.state = 'BREAK THE STATE'

AttributeError: property 'state' of 'Agent' object has no setter

The assistant message has all the extras the loop adds:
- `usage` — token counts from litellm
- `stop_reason` — "stop" (normal), "tool_calls", "error", "aborted"
- `timestamp` — Unix ms
- `thinking_blocks` / `reasoning_content` — None unless thinking is enabled
- `provider_specific_fields` — opaque bag from litellm

These extras are why `convert_to_llm` exists — they must be stripped before
sending messages back to the LLM.

## 3. `subscribe()` — the primary consumer API

In the loop notebook, we used `async for event in stream` to consume events.
The Agent doesn't expose the stream. Instead, you subscribe a callback:

```python
unsub = agent.subscribe(my_callback)  # returns unsubscribe function
```

The callback fires synchronously during `await agent.prompt()` — same thread,
no concurrency issues. This is how pi's agent works too.

**Why callbacks instead of async iteration?** The Agent is the sole reader of
the loop's EventStream (internal detail). External consumers get events via
subscribe — this lets multiple consumers see the same events (unlike a queue
where each item is consumed once).

In [34]:
agent = Agent(
    model=MODEL,
    system_prompt="Be concise.",
    convert_to_llm=simple_convert,
)

# Collect all events
events = []
unsub = agent.subscribe(lambda e: events.append(e))

await agent.prompt("Say hello!")


In [38]:
for e in events:
    print(e)

{'type': 'agent_start'}
{'type': 'turn_start'}
{'type': 'message_start', 'message': {'role': 'user', 'content': 'Say hello!', 'timestamp': 1772759831835}}
{'type': 'message_end', 'message': {'role': 'user', 'content': 'Say hello!', 'timestamp': 1772759831835}}
{'type': 'message_start', 'message': {'role': 'assistant', 'content': None, 'tool_calls': None}}
{'type': 'message_update', 'message': {'role': 'assistant', 'content': 'Hello! ', 'tool_calls': None}, 'delta': {'content': 'Hello! '}, 'delta_type': 'text_delta'}
{'type': 'message_update', 'message': {'role': 'assistant', 'content': 'Hello! 👋 How are you doing today', 'tool_calls': None}, 'delta': {'content': '👋 How are you doing today'}, 'delta_type': 'text_delta'}
{'type': 'message_update', 'message': {'role': 'assistant', 'content': 'Hello! 👋 How are you doing today? Is', 'tool_calls': None}, 'delta': {'content': '? Is'}, 'delta_type': 'text_delta'}
{'type': 'message_update', 'message': {'role': 'assistant', 'content': 'Hello! 👋 

Same events sequence as the raw loop:

Now let's test unsubscribe:

In [44]:
count_before = len(events)
count_before

13

In [45]:
agent._subscribers

[]

In [46]:
unsub()  # stop receiving events

In [47]:
agent._subscribers

[]

In [48]:
await agent.prompt("Say goodbye.")

count_after = len(events)
print(f"Events before unsub: {count_before}")
print(f"Events after second prompt: {count_after}")
print(f"Unsubscribe worked: {count_before == count_after}")

Events before unsub: 13
Events after second prompt: 13
Unsubscribe worked: True


In [49]:
agent.state.messages

[{'role': 'user', 'content': 'Say hello!', 'timestamp': 1772759831835},
 {'role': 'assistant',
  'content': 'Hello! 👋 How are you doing today? Is there something I can help you with?',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 15,
   'completion_tokens': 24,
   'total_tokens': 39,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772759832980},
 {'role': 'user', 'content': 'Say goodbye.', 'timestamp': 1772760121536},
 {'role': 'assistant',
  'content': 'Goodbye! 👋 Take care, and feel free to come back anytime you need help. 😊',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 45,
   'completion_tokens': 28,
   'total_tokens': 73,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772760122657}]

## 4. `prompt()` overloads

Like pi's `agent.prompt()`, ours accepts four input shapes:

| Input | What happens |
|-------|-------------|
| `prompt("string")` | Wrapped in `{"role": "user", "content": "string", "timestamp": ...}` |
| `prompt({"role": "user", ...})` | Used as-is |
| `prompt([msg1, msg2])` | Multiple messages injected |
| `prompt("text", images=[...])` | Multimodal: text + images in content array |

Let's see what each produces.

In [52]:
# Overload 1: string
agent = Agent(model=MODEL, convert_to_llm=simple_convert)
await agent.prompt("Hello from a string")
agent.messages[0]

{'role': 'user', 'content': 'Hello from a string', 'timestamp': 1772760810255}

In [53]:
# Overload 2: dict (used as-is)
agent = Agent(model=MODEL, convert_to_llm=simple_convert)
await agent.prompt(
    {"role": "user", "content": "Hello from a dict", "custom_field": "preserved"}
)
agent.messages[0]

{'role': 'user', 'content': 'Hello from a dict', 'custom_field': 'preserved'}

In [56]:
simple_convert(agent.messages)

[{'role': 'user', 'content': 'Hello from a dict'},
 {'role': 'assistant',
  'content': 'Hello! It looks like your message came through as plain text, but you mentioned "from a dict" — were you perhaps trying to send a Python dictionary, or is there something specific you meant by that? 😊\n\nFeel free to clarify and I\'ll be happy to help!'}]

In [59]:
# Overload 3: list of messages
agent = Agent(model=MODEL, convert_to_llm=simple_convert, system_prompt="Be concise.")
await agent.prompt(
    [
        {"role": "user", "content": "My name is Alice."},
        {"role": "user", "content": "What is my name?"},
    ]
)

In [62]:
agent.messages

[{'role': 'user', 'content': 'My name is Alice.'},
 {'role': 'user', 'content': 'What is my name?'},
 {'role': 'assistant',
  'content': 'Your name is **Alice**! You just told me. 😊',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 22,
   'completion_tokens': 18,
   'total_tokens': 40,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772761209187}]

In [68]:
# Overload 4: string + images (multimodal)
# Send a real image and ask the LLM about it

image_block = {
    "type": "image_url",
    "image_url": {"url": f"https://dev-dashhudson-static.s3.amazonaws.com/research/media_asset_ai_generation/experiments/urbn_flat_lay/92207174_707_b3.jpg"},
}

agent = Agent(
    model=MODEL,
    system_prompt="Be concise. One sentence max.",
    convert_to_llm=simple_convert,
)
await agent.prompt("What is in this image?", images=[image_block])

print(f"Response: {agent.messages[-1].get('content')}")

Response: A yellow quilted tote bag with a red cherry print pattern and padded shoulder straps.


In [69]:
agent.messages

[{'role': 'user',
  'content': [{'type': 'text', 'text': 'What is in this image?'},
   {'type': 'image_url',
    'image_url': {'url': 'https://dev-dashhudson-static.s3.amazonaws.com/research/media_asset_ai_generation/experiments/urbn_flat_lay/92207174_707_b3.jpg'}}],
  'timestamp': 1772797438256},
 {'role': 'assistant',
  'content': 'A yellow quilted tote bag with a red cherry print pattern and padded shoulder straps.',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 1561,
   'completion_tokens': 23,
   'total_tokens': 1584,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772797441035}]

## 5. State access

The Agent tracks state in an `AgentState` dataclass. You can inspect it at any time:

```python
agent.state.is_streaming       # True while loop is running
agent.state.stream_message     # current partial message being streamed (or None)
agent.state.pending_tool_calls # set of tool call IDs currently executing
agent.state.error              # last error message (or None)
agent.state.model              # current model string
agent.state.system_prompt      # current system prompt
agent.state.tools              # current tool list
agent.state.thinking_level     # "off", "minimal", "low", "medium", "high", "xhigh"
agent.messages                 # shorthand for agent.state.messages
```

Let's watch state change *during* a run using a subscriber:

In [ ]:
agent = Agent(
    model=MODEL,
    system_prompt="Be concise.",
    convert_to_llm=simple_convert,
    tools=[echo_tool],
)

# Track state transitions
state_log = []

print(
    f"{'Event':<26} {'Streaming':<10} {'StreamMsg':<10} {'PendTools':<10} {'MsgCount':<10}"
)
def track_state(event):
    t = event["type"]
    entry = {
        "event": t,
        "is_streaming": agent.state.is_streaming,
        "stream_msg": agent.state.stream_message is not None,
        "pending_tools": len(agent.state.pending_tool_calls),
        "msg_count": len(agent.messages),
    }
    state_log.append(entry)
    print("-" * 66)
    print(
            f"{entry['event']:<26} {str(entry['is_streaming']):<10} {str(entry['stream_msg']):<10} {entry['pending_tools']:<10} {entry['msg_count']:<10}"
        )

agent.subscribe(track_state)
await agent.prompt("Echo 'hello world'")



Event                      Streaming  StreamMsg  PendTools  MsgCount  
------------------------------------------------------------------
agent_start                True       False      0          0         
------------------------------------------------------------------
turn_start                 True       False      0          0         
------------------------------------------------------------------
message_start              True       False      0          0         
------------------------------------------------------------------
message_end                True       False      0          1         
------------------------------------------------------------------
message_start              True       True       0          1         
------------------------------------------------------------------
message_update             True       True       0          1         
------------------------------------------------------------------
message_update             True   

In [76]:
import pandas as pd
pd.DataFrame(state_log)

,event,is_streaming,stream_msg,pending_tools,msg_count
0,agent_start,True,False,0,0
1,turn_start,True,False,0,0
2,message_start,True,False,0,0
3,message_end,True,False,0,1
4,message_start,True,True,0,1
5,message_update,True,True,0,1
6,message_update,True,True,0,1
7,message_update,True,True,0,1
8,message_update,True,True,0,1
9,message_update,True,True,0,1


Notice how:
- `is_streaming` is True throughout the run
- `stream_message` appears on `message_start` (assistant only), disappears on `message_end`
- `pending_tool_calls` increments on `tool_execution_start`, decrements on `tool_execution_end`
- `msg_count` grows on each `message_end` — messages are appended incrementally, not batched


The tool call produces this pattern:
```
Turn 1: assistant calls echo → tool executes → tool result
Turn 2: assistant sees result → responds with text
```

Messages: user → assistant (tool_call) → tool (result) → assistant (text response)

## 6. Multi-turn — why Agent is stateful

This is the Agent's main value: messages persist across `prompt()` calls.
With the raw loop, you'd need to manually thread context between calls.
The Agent does it automatically.

In [80]:
agent = Agent(
    model=MODEL,
    system_prompt="Be concise. Remember everything the user says.",
    convert_to_llm=simple_convert,
)

# Turn 1: tell the agent something
await agent.prompt("My favorite color is blue.")
agent.messages



[{'role': 'user',
  'content': 'My favorite color is blue.',
  'timestamp': 1772799019275},
 {'role': 'assistant',
  'content': "Got it! Blue is a great color. I'll remember that your favorite color is blue.",
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 24,
   'completion_tokens': 22,
   'total_tokens': 46,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772799020497}]

In [81]:
# Turn 2: ask about it — the agent should remember
await agent.prompt("What is my favorite color?")
agent.messages

[{'role': 'user',
  'content': 'My favorite color is blue.',
  'timestamp': 1772799019275},
 {'role': 'assistant',
  'content': "Got it! Blue is a great color. I'll remember that your favorite color is blue.",
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 24,
   'completion_tokens': 22,
   'total_tokens': 46,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772799020497},
 {'role': 'user',
  'content': 'What is my favorite color?',
  'timestamp': 1772799034426},
 {'role': 'assistant',
  'content': 'Your favorite color is blue!',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 55,
   'completion_tokens': 9,
   'total_tokens': 64,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772799035349}]

## 7. Steering — `steer()` mid-run

`steer()` queues a message that gets injected **during** a run:
- After each tool execution, the loop checks the steering queue
- If there's a message, remaining tools are **skipped** and the steering message
  is injected before the next LLM call
- This is "stop what you're doing, do this instead"

The loop also checks for steering at the start of each run (before the first LLM call).
So if you call `steer()` before `prompt()`, the steering message gets picked up immediately.

Let's demonstrate both: pre-queued steering, and mid-tool steering.

In [82]:
# Pre-queued steering: steer() before prompt()
agent = Agent(
    model=MODEL,
    system_prompt="Be concise.",
    convert_to_llm=simple_convert,
)

agent.steer("Actually, tell me a joke instead.")
await agent.prompt("What is the capital of France?")

agent.messages

[{'role': 'user',
  'content': 'What is the capital of France?',
  'timestamp': 1772799168810},
 {'role': 'user',
  'content': 'Actually, tell me a joke instead.',
  'timestamp': 1772799168810},
 {'role': 'assistant',
  'content': "Here's one:\n\nWhy don't scientists trust atoms?\n\n**Because they make up everything!** 😄",
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 27,
   'completion_tokens': 27,
   'total_tokens': 54,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772799170143}]

In [83]:
# Mid-tool steering: steer() during tool execution
# When tool_a executes, it queues a steering message.
# tool_b should be SKIPPED.

call_log = []
steering_agent = None  # forward reference


async def tool_a_exec(tool_call_id, params, signal=None, on_update=None):
    call_log.append("a")
    steering_agent.steer("Stop! Do something else.")  # interrupt!
    return ToolResult(content=[{"type": "text", "text": "tool_a done"}])


async def tool_b_exec(tool_call_id, params, signal=None, on_update=None):
    call_log.append("b")
    return ToolResult(content=[{"type": "text", "text": "tool_b done"}])


tool_a = Tool(
    name="tool_a",
    description="Tool A",
    parameters={"type": "object", "properties": {}},
    execute=tool_a_exec,
)
tool_b = Tool(
    name="tool_b",
    description="Tool B",
    parameters={"type": "object", "properties": {}},
    execute=tool_b_exec,
)

steering_agent = Agent(
    model=MODEL,
    system_prompt="When asked, call both tool_a and tool_b in a single response. Be concise.",
    convert_to_llm=simple_convert,
    tools=[tool_a, tool_b],
)

await steering_agent.prompt("Call both tool_a and tool_b now.")

steering_agent.messages

[{'role': 'user',
  'content': 'Call both tool_a and tool_b now.',
  'timestamp': 1772799262271},
 {'role': 'assistant',
  'content': 'Sure! Calling both tools simultaneously right away!',
  'tool_calls': [{'id': 'toolu_015hmTYvLiFfgtrXCfVhJ7UQ',
    'type': 'function',
    'function': {'name': 'tool_a', 'arguments': '{}'},
    'provider_specific_fields': None},
   {'id': 'toolu_01HADL4L2pYRrLmFePX8mg6j',
    'type': 'function',
    'function': {'name': 'tool_b', 'arguments': '{}'},
    'provider_specific_fields': None}],
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 609,
   'completion_tokens': 66,
   'total_tokens': 675,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'tool_calls',
  'timestamp': 1772799264200},
 {'role': 'tool',
  'tool_call_id': 'toolu_015hmTYvLiFfgtrXCfVhJ7UQ',
  'name': 'tool_a',
  'content': [{'type': 'text', 'text': 'tool_a done'}],
  'details': {},
  'is_erro

  1. Assistant calls both tool_a and tool_b
  2. tool_a executes (and queues steering inside its execute function)
  3. tool_b gets skipped — is_error: True, "Skipped due to queued user message."
  4. Steering message injected: "Stop! Do something else."
  5. Assistant responds to the steering instead of continuing

  The key proof: tool_b never ran ('b' not in call_log), but it still has a tool result in the conversation — that's the synthetic skip result from _skip_tool_call() in
  loop.py:94-129. The LLM needs every tool call to have a result, even skipped ones.

## 8. Follow-up — `follow_up()` after idle

`follow_up()` is the *outer loop* mechanism. Unlike steering (which interrupts),
follow-ups wait until the agent finishes everything (no more tool calls, no steering).
Then the follow-up message is injected and the agent continues.

- Steering = "stop what you're doing" (immediate)
- Follow-up = "when you're done, also do this" (deferred)

In [84]:
agent = Agent(
    model=MODEL,
    system_prompt="Be concise. One sentence.",
    convert_to_llm=simple_convert,
)

# Queue a follow-up BEFORE the first prompt
agent.follow_up("Now tell me a fun fact about cats.")

await agent.prompt("What is 2 + 2?")

agent.messages

[{'role': 'user', 'content': 'What is 2 + 2?', 'timestamp': 1772799462214},
 {'role': 'assistant',
  'content': '4',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 24,
   'completion_tokens': 5,
   'total_tokens': 29,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772799463351},
 {'role': 'user',
  'content': 'Now tell me a fun fact about cats.',
  'timestamp': 1772799462213},
 {'role': 'assistant',
  'content': 'Cats spend about 70% of their lives sleeping.',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 41,
   'completion_tokens': 15,
   'total_tokens': 56,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772799464442}]

## 9. Queue modes

Both steering and follow-up have two modes:
- `"one-at-a-time"` (default) — dequeue one message per poll
- `"all"` — dequeue everything at once

This matters when multiple messages are queued. Let's see the difference.

In [86]:
# one-at-a-time (default): queue 3, dequeue returns 1
agent = Agent(model=MODEL, convert_to_llm=simple_convert)
agent.steer("msg1")
agent.steer("msg2")
agent.steer("msg3")

batch = agent._dequeue_steering()
print(
    f"one-at-a-time: got {len(batch)} message(s), {len(agent._steering_queue)} remaining"
)
print(f"  dequeued: '{batch[0]['content']}'")

one-at-a-time: got 1 message(s), 2 remaining
  dequeued: 'msg1'


In [87]:
# all mode: queue 3, dequeue returns all 3
agent = Agent(model=MODEL, convert_to_llm=simple_convert, steering_mode="all")
agent.steer("msg1")
agent.steer("msg2")
agent.steer("msg3")

batch = agent._dequeue_steering()
print(f"all mode: got {len(batch)} message(s), {len(agent._steering_queue)} remaining")
for m in batch:
    print(f"  '{m['content']}'")

all mode: got 3 message(s), 0 remaining
  'msg1'
  'msg2'
  'msg3'


## 10. `continue_run()` — resume from context

`continue_run()` is for when the conversation ended at a tool result or user message
and you want the LLM to continue from there — without sending a new prompt.

Three interesting cases when the last message is an assistant message:
1. Steering queue has messages → use those
2. Follow-up queue has messages → use those
3. Both empty → error (can't continue from assistant without new input)

In [ ]:
# Case: continue from a tool result (manually built context)
agent = Agent(
    model=MODEL,
    system_prompt="Be concise.",
    convert_to_llm=simple_convert,
)

# Simulate: user asked about weather → assistant called tool → we have the result
agent._state.messages = [
    {"role": "user", "content": "What's the weather?"},
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [
            {"id": "c0", "function": {"name": "weather", "arguments": "{}"}}
        ],
        "stop_reason": "tool_calls",
    },
    {
        "role": "tool",
        "tool_call_id": "c0",
        "content": [{"type": "text", "text": "72°F and sunny in San Francisco"}],
        "is_error": False,
    },
]

await agent.continue_run()

# The LLM should have summarized the weather
last = [m for m in agent.messages if m.get("role") == "assistant"][-1]
print(f"Assistant response: {(last.get('content') or '')[:100]}")

In [ ]:
# Case: continue from assistant + steering queue
agent = Agent(model=MODEL, convert_to_llm=simple_convert)
agent._state.messages = [
    {"role": "user", "content": "Hi"},
    {"role": "assistant", "content": "Hello!", "stop_reason": "stop"},
]
agent.steer("Now tell me a joke.")

await agent.continue_run()

last = [m for m in agent.messages if m.get("role") == "assistant"][-1]
print(f"Assistant (after steering): {(last.get('content') or '')[:100]}")

In [ ]:
# Case: continue from assistant + empty queues → error
agent = Agent(model=MODEL, convert_to_llm=simple_convert)
agent._state.messages = [
    {"role": "user", "content": "Hi"},
    {"role": "assistant", "content": "Hello!", "stop_reason": "stop"},
]

try:
    await agent.continue_run()
except ValueError as e:
    print(f"Got expected error: {e}")

## 12. `abort()` and partial preservation

`abort()` sets a signal that stops the loop. But what happens to the partial
assistant message that was being streamed? The Agent handles this edge case
(same as pi's agent.ts lines 504-518):

1. If the partial has **real content** (non-empty text, reasoning, or named tool call) → **preserve it**
2. If it's just **empty scaffolding** (empty strings, unnamed tool calls) → **discard it**
3. If discarded after abort → raise "Request was aborted" → caught by error handler

In [ ]:
# abort() after receiving some text — partial should be preserved
agent = Agent(
    model=MODEL,
    system_prompt="Write a very long essay about the history of computing. At least 5000 words.",
    convert_to_llm=simple_convert,
)

chunk_count = 0


def abort_after_chunks(event):
    global chunk_count
    if event["type"] == "message_update" and event.get("delta_type") == "text_delta":
        chunk_count += 1
        if chunk_count >= 5:
            agent.abort()


agent.subscribe(abort_after_chunks)
await agent.prompt("Go ahead.")

print(f"is_streaming: {agent.state.is_streaming}")
print(f"signal cleaned up: {agent._signal is None}")

# Check what we got
assistants = [m for m in agent.messages if m.get("role") == "assistant"]
if assistants:
    last = assistants[-1]
    content = last.get("content") or ""
    print(f"Got {len(content)} chars of partial content")
    print(f"stop_reason: {last.get('stop_reason')}")
    print(f"First 100 chars: {content[:100]}...")
else:
    print(f"No assistant message (error: {agent.state.error})")

## 13. `wait_for_idle()`

Returns immediately when the agent isn't running. Blocks until the current run completes.
Useful when you fire-and-forget a prompt and need to sync later.

In [ ]:
# When idle, returns immediately
agent = Agent(model=MODEL, convert_to_llm=simple_convert)
await agent.wait_for_idle()  # should not hang
print("wait_for_idle() returned immediately (agent is idle)")

# After a prompt, also returns immediately (prompt already blocks)
await agent.prompt("Hi")
await agent.wait_for_idle()
print("wait_for_idle() returned immediately (prompt already completed)")

## 14. `reset()` vs `clear_messages()`

Two ways to clear state, with different scopes:

| Method | Clears messages | Clears queues | Clears error | Keeps config |
|--------|:-:|:-:|:-:|:-:|
| `reset()` | ✓ | ✓ | ✓ | ✓ |
| `clear_messages()` | ✓ | ✗ | ✗ | ✓ |

`reset()` is "start over". `clear_messages()` is "clear history but keep queued work".

In [ ]:
agent = Agent(
    model=MODEL,
    system_prompt="sys",
    tools=[echo_tool],
    convert_to_llm=simple_convert,
)
agent.append_message({"role": "user", "content": "old message"})
agent.steer("queued steering")
agent.follow_up("queued follow-up")
agent._state.error = "some error"

print("Before clear_messages():")
print(
    f"  messages: {len(agent.messages)}, queued: {agent.has_queued_messages()}, error: {agent.state.error}"
)

agent.clear_messages()

print("After clear_messages():")
print(
    f"  messages: {len(agent.messages)}, queued: {agent.has_queued_messages()}, error: {agent.state.error}"
)
print("  ↑ messages cleared, but queues and error preserved")

In [ ]:
# Now reset — clears everything
agent.append_message({"role": "user", "content": "new message"})
agent.reset()

print("After reset():")
print(
    f"  messages: {len(agent.messages)}, queued: {agent.has_queued_messages()}, error: {agent.state.error}"
)
print(f"  model: {agent.state.model}  ← preserved")
print(f"  system_prompt: '{agent.state.system_prompt}'  ← preserved")
print(f"  tools: {len(agent.state.tools)}  ← preserved")

## 15. Configuration setters — mid-run changes

Pi allows calling `setModel()`, `setTools()`, etc. even while the agent is streaming.
The loop snapshots context at the start of each run, so mid-run changes only take
effect on the **next** run. We match this behavior — no streaming guard.

This enables patterns like:
- **Dynamic capabilities:** escalate tool permissions mid-conversation
- **Model switching:** use a cheap model for simple tasks, expensive for complex ones
- **Plan mode:** swap tool sets between planning and execution phases

In [ ]:
agent = Agent(model=MODEL, convert_to_llm=simple_convert)

# Track which model gets called
events = []
agent.subscribe(lambda e: events.append(e))

await agent.prompt("Say 'model A'.")
first_model = [m for m in agent.messages if m.get("role") == "assistant"][-1].get(
    "model", "?"
)

# Switch model mid-conversation
agent.set_model("gemini/gemini-3-flash-preview")
assert agent.state.model == "gemini/gemini-3-flash-preview"

await agent.prompt("Say 'model B'.")

# Show both models were used
assistants = [m for m in agent.messages if m.get("role") == "assistant"]
for a in assistants:
    print(
        f"  content='{(a.get('content') or '')[:30]}' | stop_reason={a.get('stop_reason')}"
    )

## 16. Error handling

When the LLM call fails (network error, rate limit, etc.), the Agent:
1. Catches the exception
2. Creates a synthetic assistant message with `stop_reason="error"`
3. Appends it to messages
4. Sets `agent.state.error`
5. Emits `agent_end` event
6. Cleans up (is_streaming=False, etc.)

The Agent does **not** re-raise — it always completes cleanly. This lets consumers
check `agent.state.error` instead of wrapping every `prompt()` in try/except.

In [ ]:
# Force an error by using a non-existent model
agent = Agent(model="fake-provider/nonexistent-model", convert_to_llm=simple_convert)
await agent.prompt("This will fail.")

print(f"is_streaming: {agent.state.is_streaming}")
print(f"error: {agent.state.error[:80]}...")

# The error message has a specific structure
error_msg = [
    m for m in agent.messages if m.get("role") == "assistant" and m.get("error_message")
][-1]
print("\nError message structure:")
print(f"  role: {error_msg['role']}")
print(f"  stop_reason: {error_msg['stop_reason']}")
print(f"  error_message: {error_msg['error_message'][:80]}...")
print(f"  model: {error_msg.get('model')}")
print(f"  usage: {error_msg.get('usage')}")
print(f"  timestamp: {error_msg.get('timestamp')}")

## 17. `_default_convert_to_llm`

If you don't provide `convert_to_llm`, the Agent uses a built-in default.
Let's see what it does vs what we'd need for specific providers.

In [ ]:
from liteagent.agent import _default_convert_to_llm

# Simulate a conversation with enriched messages
messages = [
    {"role": "user", "content": "Hi", "timestamp": 12345},
    {
        "role": "assistant",
        "content": "Hello!",
        "tool_calls": None,
        "thinking_blocks": None,
        "reasoning_content": None,
        "usage": {"prompt_tokens": 10, "completion_tokens": 5},
        "stop_reason": "stop",
        "timestamp": 12346,
        "provider_specific_fields": {"some": "thing"},
    },
    {
        "role": "tool",
        "tool_call_id": "c0",
        "content": [{"type": "text", "text": "result"}],
        "name": "echo",
        "is_error": False,
        "details": {"extra": "data"},
        "timestamp": 12347,
    },
]

converted = _default_convert_to_llm(messages)
print("After _default_convert_to_llm:")
for m in converted:
    print(f"\n  {m}")

print("\nWhat was stripped:")
print("  - user: timestamp")
print("  - assistant: usage, stop_reason, timestamp, provider_specific_fields")
print("  - tool: name, is_error, details, timestamp + content flattened to string")

The default `convert_to_llm` is good enough for most cases. You'd override it when:

1. **Custom message types** — your app stores `{"role": "notification", ...}` that need
   to be filtered out or converted to user messages
2. **Provider quirks** — OpenAI requires tool result content as a plain string, not
   a content block array (the default already handles this for tool messages)
3. **Multimodal tool results** — images in tool results need to be routed to a
   follow-up user message for providers that don't support images in tool results

## 18. Testing across models

The Agent is model-agnostic. Let's run the same prompt + tool call through
all 4 active target models.

In [ ]:
MODELS = [
    "anthropic/claude-sonnet-4-6",
    "anthropic/claude-opus-4-6",
    "gemini/gemini-3-flash-preview",
    "gpt-5.2",
]

for model in MODELS:
    agent = Agent(
        model=model,
        system_prompt="Use the echo tool. Be concise.",
        convert_to_llm=simple_convert,
        tools=[echo_tool],
    )

    try:
        await agent.prompt("Echo 'test'")
        roles = [m.get("role") for m in agent.messages]
        has_tool = "tool" in roles
        assistants = [m for m in agent.messages if m.get("role") == "assistant"]
        last_content = (assistants[-1].get("content") or "")[:50] if assistants else "?"
        print(f"  ✓ {model:<42} tool_used={has_tool} | {last_content}")
    except Exception as e:
        print(f"  ✗ {model:<42} ERROR: {e}")

## 19. Real-world patterns

The Agent is framework-agnostic. Here's how you'd wire it into different consumers.

In [ ]:
# Pattern 1: CLI — print text deltas as they arrive

agent = Agent(
    model=MODEL,
    system_prompt="Be concise.",
    convert_to_llm=simple_convert,
)


def cli_handler(event):
    if event["type"] == "message_update" and event.get("delta_type") == "text_delta":
        text = event["delta"].get("content", "")
        print(text, end="", flush=True)
    elif event["type"] == "agent_end":
        print()  # newline at the end


agent.subscribe(cli_handler)
print("Agent: ", end="")
await agent.prompt("What is the meaning of life, in one sentence?")

In [ ]:
# Pattern 2: Event collector for SSE/WebSocket
# In a real app, you'd push to an asyncio.Queue that an SSE endpoint drains
import asyncio

agent = Agent(
    model=MODEL,
    system_prompt="Be concise.",
    convert_to_llm=simple_convert,
)

sse_queue = asyncio.Queue()
agent.subscribe(lambda e: sse_queue.put_nowait(e))

# In a real app: asyncio.create_task(agent.prompt("..."))
# Then an SSE endpoint would: while True: event = await sse_queue.get(); yield event
await agent.prompt("Hello")

# Drain the queue to see what SSE clients would receive
sse_events = []
while not sse_queue.empty():
    sse_events.append(sse_queue.get_nowait())

print(f"SSE events: {len(sse_events)}")
print(f"Types: {[e['type'] for e in sse_events[:5]]}... (showing first 5)")

## 20. Summary

### Agent vs raw loop — when to use which

| Use Agent when... | Use raw loop when... |
|---|---|
| Building a chat interface | You need full control over context threading |
| You want stateful conversations | You're building a one-shot pipeline |
| You need steering/follow-up queues | You want to manage your own EventStream |
| You want subscribe-based event delivery | You prefer `async for` iteration |
| You want abort/reset/wait_for_idle | You're embedding in an existing event loop |

### What the Agent adds over the raw loop

```
Raw loop (loop.py)              Agent (agent.py)
─────────────────               ───────────────────
Stateless                       Stateful (messages persist)
Returns EventStream             subscribe() callbacks
You manage context              Context managed automatically
You handle cancellation         abort() + partial preservation
No queues                       steer() + follow_up() with modes
```

### Comparison with pi's agent.ts

Our Agent is a faithful port at ~95% fidelity. Same methods, same state management,
same event handling, same partial preservation logic. The differences are:
- We use plain dicts instead of typed message classes
- We skip pi-specific config (sessionId, streamFn, transport) — litellm handles those
- We use `asyncio.Event` instead of `AbortController` for cancellation

See `COMPARISONS.md` for the full fidelity scorecard and design decisions.